# DecisionTreeClassifier

Pipeline hoàn chỉnh trong một notebook:
[Dữ liệu thô: .csv] 
       │
       ▼
 1. ĐỌC & LÀM SẠCH (load_dataset)
       │
       ▼
 2. TIỀN XỬ LÝ (Preprocessor)   
       │  ├─ Imputer (Điền giá trị thiếu)
       │  └─ Scaler (Đưa số về khoảng 0-1)
       │
       ▼
 3. VÒNG LẶP TUNING (PARAM_GRID) ───► Thử nghiệm tìm cấu hình tối ưu nhất
       │
       ▼
 4. RE-FIT & ĐÁNH GIÁ (Test)     ───► Test nghiệm thu cuối cùng (Feature Test)
       │
       ▼
 5. LƯU TRỮ (.joblib) 

In [6]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.tree import DecisionTreeClassifier

MODEL_NAME = "DecisionTreeClassifier"
TARGET_COLUMN = "target"
DATA_DIR = Path("../data")
ARTIFACTS_DIR = Path("../notebook_outputs")
#tham số thử nghiệm
PARAM_GRID = [
    {'max_depth': 3, 'min_samples_split': 2, 'min_samples_leaf': 1},
    {'max_depth': 4, 'min_samples_split': 4, 'min_samples_leaf': 2},
    {'max_depth': 5, 'min_samples_split': 6, 'min_samples_leaf': 2},
    {'max_depth': 7, 'min_samples_split': 10, 'min_samples_leaf': 4}
]

#1. Khởi tạo cấu hình và Tiền xử lý dữ liệu (Data Preprocessing)
def parse_numeric_value(value):
    if pd.isna(value):
        return value
    if isinstance(value, str):
        cleaned = value.strip()
        if cleaned.count(".") > 1:
            sign = -1 if cleaned.startswith("-") else 1
            digits_only = cleaned[1:] if sign == -1 else cleaned
            digits_only = digits_only.replace(".", "")
            return sign * float(f"0.{digits_only}")
        return float(cleaned)
    return float(value)

def load_dataset(file_path):
    df = pd.read_csv(file_path)
    object_columns = df.select_dtypes(include="object").columns
    for col in object_columns:
        df[col] = df[col].map(parse_numeric_value)
    return df

#Chia tập dữ liệu làm hai phần: x (các thuộc tính đầu vào như age, chol, cp...) và y (nhãn mục tiêu target - 0: không bệnh, 1: có bệnh).
def split_xy(df):
    x = df.drop(columns=[TARGET_COLUMN]).copy()
    y = df[TARGET_COLUMN].copy()
    return x, y

#: Điền các giá trị bị thiếu (NaN) bằng giá trị trung vị của cột đó.
def build_preprocessor(feature_names):
    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", MinMaxScaler()),
        ]
    )
    return ColumnTransformer(
        transformers=[("num", numeric_pipeline, feature_names)],
        remainder="drop",
    )

#2. Thiết lập cấu trúc Mô hình (Pipeline)
def build_model(params):
    estimator = clone(DecisionTreeClassifier(random_state=42)).set_params(**params)
    return Pipeline(
        steps=[
            ("preprocess", build_preprocessor(feature_names)),
            ("model", estimator),
        ]
    )


def get_scores(model, x):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(x)[:, 1]
    if hasattr(model, "decision_function"):
        return model.decision_function(x)
    return None


def evaluate_model(model, x, y):
    y_pred = model.predict(x)
    y_score = get_scores(model, x)
    metrics = {
        "accuracy": float(accuracy_score(y, y_pred)),
        "precision": float(precision_score(y, y_pred, zero_division=0)),
        "recall": float(recall_score(y, y_pred, zero_division=0)),
        "f1": float(f1_score(y, y_pred, zero_division=0)),
        "confusion_matrix": confusion_matrix(y, y_pred).tolist(),
    }
    if y_score is not None and len(np.unique(y)) > 1:
        metrics["roc_auc"] = float(roc_auc_score(y, y_score))
    return metrics

print("--- Đang nạp dữ liệu... ---")
train_df = load_dataset(DATA_DIR / "train.csv")

print("-> Đã nạp xong tập Train")
val_df = load_dataset(DATA_DIR / "val.csv")

test_df = load_dataset(DATA_DIR / "test.csv")
print("--- Đã nạp xong toàn bộ dữ liệu! ---")

x_train, y_train = split_xy(train_df)
x_val, y_val = split_xy(val_df)
x_test, y_test = split_xy(test_df)
feature_names = x_train.columns.tolist()

best_result = None

#3. Tìm kiếm Siêu tham số tối ưu (Hyperparameter Tuning)
#Đo đạc các chỉ số đánh giá (evaluate_model) bao gồm: Accuracy (Độ chính xác), Precision (Độ chuẩn xác), Recall (Độ nhạy), F1-score trên cả tập Train và tập Validation (val.csv).
#So sánh dựa trên ưu tiên: F1-score cao nhất, nếu F1-score bằng nhau thì chọn Accuracy cao nhất trên tập Validation để tìm ra bộ best_result.
for i, params in enumerate(PARAM_GRID):
    print(f"-> Đang huấn luyện bộ tham số thứ {i+1}/3: {params}")
    pipeline = build_model(params)
    pipeline.fit(x_train, y_train)
    train_metrics = evaluate_model(pipeline, x_train, y_train)
    val_metrics = evaluate_model(pipeline, x_val, y_val)
    current_result = {
        "params": params,
        "train_metrics": train_metrics,
        "val_metrics": val_metrics,
        "model": pipeline,
    }
    if best_result is None:
        best_result = current_result
    else:
        current_key = (current_result["val_metrics"]["f1"], current_result["val_metrics"]["accuracy"])
        best_key = (best_result["val_metrics"]["f1"], best_result["val_metrics"]["accuracy"])
        if current_key > best_key:
            best_result = current_result

print("Best params on validation:", best_result["params"])
print("Train metrics:", json.dumps(best_result["train_metrics"], indent=2))
print("Validation metrics:", json.dumps(best_result["val_metrics"], indent=2))

x_train_val = pd.concat([x_train, x_val], ignore_index=True)
y_train_val = pd.concat([y_train, y_val], ignore_index=True)

#4. Huấn luyện lại và Kiểm thử (Refit & Test)
final_model = build_model(best_result["params"])
final_model.fit(x_train_val, y_train_val)

train_val_metrics = evaluate_model(final_model, x_train_val, y_train_val)
test_metrics = evaluate_model(final_model, x_test, y_test)
test_predictions = final_model.predict(x_test)

models_dir = ARTIFACTS_DIR / "models"
reports_dir = ARTIFACTS_DIR / "reports"
predictions_dir = ARTIFACTS_DIR / "predictions"
models_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)
predictions_dir.mkdir(parents=True, exist_ok=True)

#5. Lưu trữ sản phẩm (Artifacts Export)
model_path = models_dir / f"{MODEL_NAME}.joblib"
report_path = reports_dir / f"{MODEL_NAME}_metrics.json"
prediction_path = predictions_dir / f"{MODEL_NAME}_test_predictions.csv"

joblib.dump(final_model, model_path)

prediction_df = x_test.copy()
prediction_df["actual_target"] = y_test.values
prediction_df["predicted_target"] = test_predictions
prediction_df.to_csv(prediction_path, index=False)

report = {
    "model_name": MODEL_NAME,
    "best_validation_params": best_result["params"],
    "train_metrics": best_result["train_metrics"],
    "validation_metrics": best_result["val_metrics"],
    "train_val_metrics_after_refit": train_val_metrics,
    "test_metrics": test_metrics,
    "saved_model_path": str(model_path),
    "prediction_path": str(prediction_path),
}

report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")

print("\nTrain+Val metrics after refit:")
print(json.dumps(train_val_metrics, indent=2))
print("\nTest metrics:")
print(json.dumps(test_metrics, indent=2))
print("\nSaved model:", model_path)
print("Saved report:", report_path)
print("Saved predictions:", prediction_path)

prediction_df.head()


--- Đang nạp dữ liệu... ---
-> Đã nạp xong tập Train
--- Đã nạp xong toàn bộ dữ liệu! ---
-> Đang huấn luyện bộ tham số thứ 1/3: {'max_depth': 3, 'min_samples_split': 2, 'min_samples_leaf': 1}
-> Đang huấn luyện bộ tham số thứ 2/3: {'max_depth': 4, 'min_samples_split': 4, 'min_samples_leaf': 2}
-> Đang huấn luyện bộ tham số thứ 3/3: {'max_depth': 5, 'min_samples_split': 6, 'min_samples_leaf': 2}
-> Đang huấn luyện bộ tham số thứ 4/3: {'max_depth': 7, 'min_samples_split': 10, 'min_samples_leaf': 4}
Best params on validation: {'max_depth': 3, 'min_samples_split': 2, 'min_samples_leaf': 1}
Train metrics: {
  "accuracy": 0.8429752066115702,
  "precision": 0.8686868686868687,
  "recall": 0.7747747747747747,
  "f1": 0.819047619047619,
  "confusion_matrix": [
    [
      118,
      13
    ],
    [
      25,
      86
    ]
  ],
  "roc_auc": 0.9073653806478236
}
Validation metrics: {
  "accuracy": 0.9333333333333333,
  "precision": 0.9285714285714286,
  "recall": 0.9285714285714286,
  "f1": 0.9

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_1280\698909956.py:43: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_columns = df.select_dtypes(include="object").columns
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_1280\698909956.py:43: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_g


Train+Val metrics after refit:
{
  "accuracy": 0.8529411764705882,
  "precision": 0.8761061946902655,
  "recall": 0.792,
  "f1": 0.8319327731092437,
  "confusion_matrix": [
    [
      133,
      14
    ],
    [
      26,
      99
    ]
  ],
  "roc_auc": 0.9117278911564626
}

Test metrics:
{
  "accuracy": 0.8064516129032258,
  "precision": 0.7857142857142857,
  "recall": 0.7857142857142857,
  "f1": 0.7857142857142857,
  "confusion_matrix": [
    [
      14,
      3
    ],
    [
      3,
      11
    ]
  ],
  "roc_auc": 0.8046218487394958
}

Saved model: ..\notebook_outputs\models\DecisionTreeClassifier.joblib
Saved report: ..\notebook_outputs\reports\DecisionTreeClassifier_metrics.json
Saved predictions: ..\notebook_outputs\predictions\DecisionTreeClassifier_test_predictions.csv


,age,trestbps,chol,thalach,oldpeak,sex,cp,fbs,restecg,exang,slope,ca,thal,actual_target,predicted_target
0,0.384303,-0.168240,-0.641646,-0.837597,0.107158,1.0,1.000000,0.0,1.0,1.0,0.5,1.0,1.0,1,1
1,-0.228879,-0.736870,-0.128635,0.106174,-0.891627,1.0,0.000000,0.0,1.0,0.0,0.0,0.0,0.0,0,0
2,0.829818,-0.545134,-0.357219,-0.175039,0.714629,1.0,0.666667,0.0,0.0,0.0,0.5,1.0,1.0,0,1
3,-0.395349,-0.545134,0.116827,-0.425278,-0.445445,0.0,0.666667,0.0,1.0,0.0,0.0,0.0,0.0,0,0
4,-0.139776,-0.623144,-0.186562,0.194515,-0.177735,1.0,0.666667,1.0,0.0,0.0,1.0,0.0,1.0,0,0


In [ ]:
#